# Thumbnail Generator
Interactive version of `src/thumbnail.py`. All config is read from `.env`.
Run each cell in order.

## 1. Setup & Imports

In [ ]:
import sys
import os
from pathlib import Path
from dotenv import load_dotenv
from IPython.display import display

ROOT = Path(".").resolve().parent
sys.path.insert(0, str(ROOT / "src"))

load_dotenv(ROOT / ".env")
print("Environment loaded from .env")
print(f"ANTHROPIC_API_KEY set : {'ANTHROPIC_API_KEY' in os.environ}")
print(f"OPENAI_API_KEY set    : {'OPENAI_API_KEY' in os.environ}")

## 2. Load & Review Configuration

In [ ]:
keys = [
    "COMPANY_NAME", "COMPANY_URL", "COMPANY_EMAIL",
    "DEFAULT_VIBE", "TITLE", "SUBTITLE",
    "CLAUDE_MODEL", "IMAGE_WIDTH", "IMAGE_HEIGHT",
]
for k in keys:
    print(f"  {k:<22} = {os.getenv(k, '(not set)')}")

## 3. Resolve Inputs (prompt for anything empty)

In [ ]:
def get_or_prompt(env_key, label, default=""):
    value = os.getenv(env_key, default).strip()
    if not value:
        value = input(f"Enter {label}: ").strip()
    return value

title         = get_or_prompt("TITLE",         "title")
subtitle      = get_or_prompt("SUBTITLE",      "subtitle")
vibe          = get_or_prompt("DEFAULT_VIBE",  "vibe (e.g. bold energetic)")
company_name  = get_or_prompt("COMPANY_NAME",  "company name")
company_url   = get_or_prompt("COMPANY_URL",   "company URL")
company_email = get_or_prompt("COMPANY_EMAIL", "company email")
model         = os.getenv("CLAUDE_MODEL", "claude-sonnet-4-6")
width         = int(os.getenv("IMAGE_WIDTH",  "1280"))
height        = int(os.getenv("IMAGE_HEIGHT", "720"))

print(f"\nTitle    : {title}")
print(f"Subtitle : {subtitle}")
print(f"Vibe     : {vibe}")
print(f"Company  : {company_name} | {company_url} | {company_email}")
print(f"Model    : {model}")
print(f"Size     : {width}×{height}")

## 4. Generate DALL-E Prompt and Color Palette via Claude

In [ ]:
import claude_helper

print("Calling Claude for DALL-E prompt...")
dalle_prompt, usage_prompt = claude_helper.generate_dalle_prompt(title, subtitle, vibe, model=model)

print("Calling Claude for color palette...")
palette, usage_palette = claude_helper.generate_palette(vibe, model=model)

print("\n--- DALL-E Prompt ---")
print(dalle_prompt)
print("\n--- Color Palette ---")
for k, v in palette.items():
    print(f"  {k}: {v}")

print(f"\nClaude usage (prompt) : {usage_prompt}")
print(f"Claude usage (palette): {usage_palette}")

## 5. Generate AI Art with DALL-E 3

In [ ]:
import image_generator

print("Calling DALL-E 3 (~10-20 seconds)...")
ai_image = image_generator.generate(dalle_prompt, width, height)

print(f"AI image size: {ai_image.size}")
display(ai_image.resize((640, 360)))  # preview at half size

## 6. Composite — Add Title, Subtitle, and Footer

In [ ]:
import compositor

final_image = compositor.compose(
    ai_image=ai_image,
    title=title,
    subtitle=subtitle,
    palette=palette,
    company_name=company_name,
    company_url=company_url,
    company_email=company_email,
    width=width,
    height=height,
)

print("Final thumbnail preview:")
display(final_image.resize((640, 360)))

## 7. Save to output/

In [ ]:
from datetime import datetime

output_dir = ROOT / "output"
output_dir.mkdir(exist_ok=True)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_path = output_dir / f"thumbnail_{timestamp}.png"
final_image.convert("RGB").save(str(output_path), format="PNG")

print(f"Saved : {output_path}")
print(f"Vibe  : {vibe}")
print(f"DALL-E cost estimate: ~$0.04")